# B-03: Enriched-feature forecasting rerun of FC-01 (D-41)

Re-runs the FC-01 tree-model benchmark (persistence/climatology + RF/XGB, driver-conditional, direct
multi-horizon, partial-pooled) on the **enriched feature set** from `NWFP_T9_Dataset_Structure.md`:
wind-direction circular encoding, TA min/max, multi-week external-soil lags/rolling, `days_since_grazing`,
expanded calendar, 3-sensor SHF, + lagged-only FCO2 in the daily track. **Round-1 HPO** applied to the
daily-track trees (RF leaf10/max_features0.5; XGB depth2/lr0.02/400-trees); hourly keeps B01-validated settings.

- **Track A (hourly):** `forecast_features_v2.csv` (base + 7 new `fx_`). Horizons {1,6,12,24,48} h.
- **Track B (daily):** `forecast_daily_v2.csv` (guide daily aggregations + daily AR). Horizons {1,3,7,14} d.
- Same CV as FC-01: Towers 4/9 train(≤2021)/test 2022–2023; Tower 2 expanding folds. Leak-free; train
  gap-filled / eval observed. Final cell compares **B03 vs FC-01** (ΔR², Δskill).

In [1]:
from pathlib import Path
import sys, datetime, numpy as np, pandas as pd
sys.path.insert(0, "../../src")
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from evaluation.metrics import full_metrics
HOURLY=Path("../../data/Hourly"); RESULTS=Path("../../results")
ALGOS=["RF","XGB"]
HORIZONS={"A":[1,6,12,24,48],"B":[1,3,7,14]}
T2_FOLDS=[("2018-06-30","2018-07-01","2018-12-31"),("2018-12-31","2019-01-01","2019-05-31")]
DUM=["is_t2","is_t4","is_t9"]

matv2=pd.read_csv(HOURLY/"forecast_features_v2.csv",low_memory=False)
matv2["Datetime"]=pd.to_datetime(matv2["Datetime"],format="mixed")
AR_A=[c for c in matv2.columns if c.startswith("ar_")]
FX_A=[c for c in matv2.columns if c.startswith("fx")]
print("hourly v2 rows",len(matv2),"| AR_A",len(AR_A),"| FX_A",len(FX_A))

hourly v2 rows 210459 | AR_A 20 | FX_A 26


## 1  Helpers (parametrised by per-track fx list)

In [2]:
def fit_model(algo, tr, feat, track="A"):
    # Round-1 HPO (D-41): daily track (B) uses tuned configs; hourly (A) keeps the B01-validated settings.
    imp=SimpleImputer(strategy="mean"); Xi=imp.fit_transform(tr[feat].values)
    if algo=="RF":
        p=dict(min_samples_leaf=5) if track=="A" else dict(min_samples_leaf=10,max_features=0.5)
        m=RandomForestRegressor(n_estimators=500,n_jobs=-1,random_state=42,**p)
    else:
        p=dict(max_depth=6,learning_rate=0.05,n_estimators=500) if track=="A" \
          else dict(max_depth=2,learning_rate=0.02,n_estimators=400,min_child_weight=10)
        m=XGBRegressor(subsample=0.8,colsample_bytree=0.8,n_jobs=-1,random_state=42,**p)
    m.fit(Xi,tr["target"].values); return m,imp

def rmse(y,p): return float(np.sqrt(mean_squared_error(y,p)))
def metrics(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    r2=r2_score(y,p) if (len(y)>1 and np.var(y)>0) else np.nan
    return rmse(y,p), float(mean_absolute_error(y,p)), r2, float(np.mean(p-y))

def climatology(tr, te, unit):
    t=te["tower"].iloc[0]; trt=tr[tr.tower==t]
    if len(trt)<24: trt=tr
    gl=float(trt["target"].mean())
    if unit=="h":
        g=trt.assign(mo=trt.ttime.dt.month,hr=trt.ttime.dt.hour).groupby(["mo","hr"])["target"].mean()
        keys=list(zip(te.ttime.dt.month,te.ttime.dt.hour)); return np.array([g.get(k,gl) for k in keys])
    g=trt.assign(mo=trt.ttime.dt.month).groupby("mo")["target"].mean()
    return np.array([g.get(m,gl) for m in te.ttime.dt.month])

def build_hourly_tables():
    T={t: matv2[matv2.tower==t].set_index("Datetime").sort_index() for t in [2,4,9]}
    return T, AR_A, FX_A

def build_daily_tables():
    dv=pd.read_csv(HOURLY/"forecast_daily_v2.csv",low_memory=False)
    dv["Datetime"]=pd.to_datetime(dv["Datetime"],format="mixed")
    AR_B=[c for c in dv.columns if c.startswith("ar_")]
    FX_B=[c for c in dv.columns if c.startswith("fx")]
    T={t: dv[dv.tower==t].set_index("Datetime").sort_index() for t in [2,4,9]}
    return T, AR_B, FX_B

def aligned(T, h, unit, ar_cols, fx_cols):
    parts=[]; off=pd.Timedelta(hours=h) if unit=="h" else pd.Timedelta(days=h)
    for t,df in T.items():
        f=df[ar_cols+DUM].copy()
        for c in fx_cols: f[c]=df[c].shift(-h)
        f["target"]=df["y_observed"].shift(-h); f["persist"]=df["y_gapfilled"]
        f["tower"]=t; f["ttime"]=df.index+off
        parts.append(f)
    return pd.concat(parts)

## 2  Run one horizon (towers 4/9 split + Tower 2 expanding folds)

In [3]:
def emit(track,h,t,bydict):
    rp=rmse(*bydict["persist"]); rc=rmse(*bydict["clim"]); rows=[]
    y_naive=bydict["persist"][1]
    for model,(y,p) in bydict.items():
        r,mae,r2,mbe=metrics(y,p)
        fm=full_metrics(y,p,y_naive=y_naive)
        rows.append(dict(track=track,horizon=h,tower=t,model=model,RMSE=round(r,3),MAE=round(mae,3),
            R2=(round(r2,3) if np.isfinite(r2) else np.nan),MBE=round(mbe,3),n_test=len(y),
            skill_persist=round(1-r/rp,3) if rp>0 else np.nan,
            skill_clim=round(1-r/rc,3) if rc>0 else np.nan,
            WAPE=round(fm["WAPE"],4) if np.isfinite(fm["WAPE"]) else np.nan,
            MASE=round(fm["MASE"],4) if np.isfinite(fm["MASE"]) else np.nan,
            sMAPE=round(fm["sMAPE"],4) if np.isfinite(fm["sMAPE"]) else np.nan,
            MAPE=round(fm["MAPE"],4) if np.isfinite(fm["MAPE"]) else np.nan,
            MAPE_n_excluded=fm["MAPE_n_excluded"]))
    return rows

def run_horizon(track, h, unit, ar_cols, fx_cols, T):
    feat=ar_cols+fx_cols+DUM; A=aligned(T,h,unit,ar_cols,fx_cols); rows=[]
    # MAIN: Towers 4 & 9 (train target<=2021, test 2022-2023)
    tr=A[A.ttime.dt.year.between(2018,2021) & A.target.notna()]
    fitted={a:fit_model(a,tr,feat,track) for a in ALGOS}
    for t in [4,9]:
        te=A[(A.tower==t)&A.ttime.dt.year.isin([2022,2023])&A.target.notna()]
        if len(te)<10: continue
        bd={"persist":(te["target"].values,te["persist"].values),
            "clim":(te["target"].values,climatology(tr,te,unit))}
        for a in ALGOS:
            m,imp=fitted[a]; bd[a]=(te["target"].values,m.predict(imp.transform(te[feat].values)))
        rows+=emit(track,h,t,bd)
    # Tower 2: expanding-window folds within 2017-2019 (donor = Tower 4)
    acc={k:([],[]) for k in ["persist","clim"]+ALGOS}
    for cut,s,e in T2_FOLDS:
        trf=A[(A.ttime<=pd.Timestamp(cut)) & A.target.notna()]
        te=A[(A.tower==2)&(A.ttime>=pd.Timestamp(s))&(A.ttime<=pd.Timestamp(e))&A.target.notna()]
        if len(te)<5 or len(trf)<50: continue
        y=te["target"].values
        acc["persist"][0].append(y); acc["persist"][1].append(te["persist"].values)
        acc["clim"][0].append(y); acc["clim"][1].append(climatology(trf,te,unit))
        for a in ALGOS:
            m,imp=fit_model(a,trf,feat,track); acc[a][0].append(y); acc[a][1].append(m.predict(imp.transform(te[feat].values)))
    if acc["persist"][0]:
        bd={k:(np.concatenate(v[0]),np.concatenate(v[1])) for k,v in acc.items()}
        rows+=emit(track,h,2,bd)
    return rows

## 3  Run both tracks, all horizons

In [4]:
TA,AR_A_,FX_A_=build_hourly_tables()
TB,AR_B_,FX_B_=build_daily_tables()
print("Track A: AR",len(AR_A_),"FX",len(FX_A_),"| Track B: AR",len(AR_B_),"FX",len(FX_B_))
ALL=[]
for h in HORIZONS["A"]:
    ALL+=run_horizon("A",h,"h",AR_A_,FX_A_,TA); print("Track A h=",h,"done",flush=True)
for h in HORIZONS["B"]:
    ALL+=run_horizon("B",h,"D",AR_B_,FX_B_,TB); print("Track B d=",h,"done",flush=True)
R=pd.DataFrame(ALL); print("rows",len(R))

Track A: AR 20 FX 26 | Track B: AR 7 FX 34


Track A h= 1 done


Track A h= 6 done


Track A h= 12 done


Track A h= 24 done


...

Track B d= 1 done


Track B d= 3 done


Track B d= 7 done


Track B d= 14 done


rows 108


## 4  Results + save b03_summary.csv

In [5]:
pd.set_option("display.width",200)
print("=== Track B (daily) — R2 (towers 4/9) ===")
print(R[(R.track=="B")&(R.tower.isin([4,9]))&(R.model.isin(["RF","XGB"]))]
      .pivot_table(index=["tower","model"],columns="horizon",values="R2").round(3).to_string())
print("\n=== Track A (hourly) — R2 (towers 4/9) ===")
print(R[(R.track=="A")&(R.tower.isin([4,9]))&(R.model.isin(["RF","XGB"]))]
      .pivot_table(index=["tower","model"],columns="horizon",values="R2").round(3).to_string())
print("\n=== skill vs persistence (RF, both tracks) ===")
print(R[R.model=="RF"].pivot_table(index=["track","tower"],columns="horizon",values="skill_persist").round(3).to_string())
R.to_csv(RESULTS/"b03_summary.csv",index=False); print("\nsaved results/b03_summary.csv")

=== Track B (daily) — R2 (towers 4/9) ===
horizon         1      3      7      14
tower model                            
4     RF     0.357  0.345  0.287  0.270
      XGB    0.362  0.313  0.284  0.259
9     RF     0.388  0.350  0.364  0.342
      XGB    0.324  0.271  0.303  0.275

=== Track A (hourly) — R2 (towers 4/9) ===
horizon         1      6      12     24     48
tower model                                   
4     RF     0.136  0.044  0.078  0.049  0.032
      XGB    0.119  0.048  0.065  0.030  0.035
9     RF     0.159  0.084  0.081  0.087  0.055
      XGB    0.132  0.042  0.021  0.064  0.052

=== skill vs persistence (RF, both tracks) ===
horizon         1      3      6      7      12     14     24     48
track tower                                                        
A     2      0.085    NaN  0.154    NaN  0.226    NaN  0.207  0.241
      4      0.110    NaN  0.210    NaN  0.195    NaN  0.182  0.141
      9      0.092    NaN  0.234    NaN  0.256    NaN  0.199  0.239
B   

## 5  Compare B03 vs FC-01 (are the enriched features an improvement?)

In [6]:
fc01=pd.read_csv(RESULTS/"fc01_summary.csv")
key=["track","tower","horizon","model"]
j=R.merge(fc01[key+["R2","skill_persist","RMSE"]],on=key,suffixes=("_b03","_fc01"))
j["dR2"]=j["R2_b03"]-j["R2_fc01"]; j["dskill"]=j["skill_persist_b03"]-j["skill_persist_fc01"]
j["dRMSE"]=j["RMSE_b03"]-j["RMSE_fc01"]
print("=== mean delta (B03 - FC-01) by track/model, towers 4/9 ===")
print(j[j.tower.isin([4,9])].groupby(["track","model"])[["dR2","dskill","dRMSE"]].mean().round(3).to_string())
print("\n=== daily h=1 R2 head-to-head (towers 4/9) ===")
print(j[(j.track=="B")&(j.horizon==1)&(j.tower.isin([4,9]))][["tower","model","R2_fc01","R2_b03","dR2"]].round(3).to_string(index=False))
print("\n=== best daily R2 per tower (any horizon/model) — FC-01 vs B03 ===")
for t in [4,9]:
    a=fc01[(fc01.track=="B")&(fc01.tower==t)&(fc01.model.isin(["RF","XGB"]))]["R2"].max()
    b=R[(R.track=="B")&(R.tower==t)&(R.model.isin(["RF","XGB"]))]["R2"].max()
    print(f"  Tower {t}: FC-01 {a:.3f}  ->  B03 {b:.3f}  (delta {b-a:+.3f})")

=== mean delta (B03 - FC-01) by track/model, towers 4/9 ===
                 dR2  dskill  dRMSE
track model                        
A     RF       0.016   0.007 -1.132
      XGB      0.036   0.015 -2.496
      clim     0.000   0.000  0.000
      persist  0.000   0.000  0.000
B     RF       0.118   0.065 -4.661
      XGB      0.166   0.085 -6.242
      clim     0.000   0.000  0.000
      persist  0.000   0.000  0.000

=== daily h=1 R2 head-to-head (towers 4/9) ===
 tower   model  R2_fc01  R2_b03   dR2
     4 persist    0.440   0.440 0.000
     4    clim   -0.022  -0.022 0.000
     4      RF    0.256   0.357 0.101
     4     XGB    0.263   0.362 0.099
     9 persist    0.231   0.231 0.000
     9    clim    0.045   0.045 0.000
     9      RF    0.304   0.388 0.084
     9     XGB    0.189   0.324 0.135

=== best daily R2 per tower (any horizon/model) — FC-01 vs B03 ===
  Tower 4: FC-01 0.263  ->  B03 0.362  (delta +0.099)
  Tower 9: FC-01 0.304  ->  B03 0.388  (delta +0.084)


## 6  Append to benchmarks.csv (B03)

In [7]:
bench=RESULTS/"benchmarks.csv"; today=datetime.date.today().isoformat()
ex=pd.read_csv(bench); ex=ex[ex["replication"]!="B03"]
rows=[]
for _,r in R.iterrows():
    rows.append({"replication":"B03","model":r["model"],"tower":f"Tower {int(r.tower)}",
        "feature_set":"enriched_v2 (guide features: WD/TA-minmax/soil-lags/days-since-grazing)","track":r["track"],
        "horizon":int(r["horizon"]),"split":"fc_main" if r.tower in (4,9) else "fc_t2folds",
        "R2":r["R2"],"RMSE":r["RMSE"],"MAE":r["MAE"],"MBE":r["MBE"],"n_test":int(r["n_test"]),
        "skill_persist":r["skill_persist"],"skill_clim":r["skill_clim"],
        "WAPE":r["WAPE"],"MASE":r["MASE"],"sMAPE":r["sMAPE"],"MAPE":r["MAPE"],"MAPE_n_excluded":r["MAPE_n_excluded"],
        "date":today,
        "notes":"B03 enriched-feature rerun of FC-01 + Round-1 daily HPO; driver-conditional; train gap-filled/eval observed (D-41); WAPE/MASE/sMAPE/MAPE added D-44b"})
new=pd.DataFrame(rows); comb=pd.concat([ex,new],ignore_index=True); comb.to_csv(bench,index=False)
print(f"Wrote {len(new)} B03 rows. Total {len(comb)}.")

Wrote 108 B03 rows. Total 3665.
